# Goal setting

In [53]:
import json

with open('gatchina.json', 'r') as file:
    analysis = json.load(file)

analysis

{'диагностика экономического и пространственного состояния МО': {'социально-экономический анализ (демография, производительность труда, структура и динамика ВГП, рынок труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность)': 'На основе анализа доступных источников представлю социально-экономический анализ Гатчинского муниципального округа Ленинградской области.\n\n## СОЦИАЛЬНО-ЭКОНОМИЧЕСКИЙ АНАЛИЗ ГАТЧИНСКОГО МУНИЦИПАЛЬНОГО ОКРУГА\n\n### ДЕМОГРАФИЯ\n\n**Численность населения:** По состоянию на 1 января 2024 года численность населения Гатчинского муниципального округа составляет 261 522 человека, что составляет 12,85% от общей численности населения Ленинградской области¹. \n\n**Структура населения:** Из общей численности сельское население составляет 89 832 человека¹. В округе расположено 234 сельских населённых пункта, крупнейшими из которых являются Новый Свет, Большие Колпаны, Малое Верево².\n\n**Городское ядро:** Административный центр — город Гатчина с населением

In [ ]:
from src import embedding
import pandas as pd
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

store = InMemoryVectorStore(embedding=embedding)
indicators_df = pd.read_excel('./data/indicators.xlsx')[['Раздел', 'Код показателя', 'Название показателя', 'Единица измерения']].copy()

docs = []
for _,row in indicators_df.iterrows():
    data = row.to_dict()
    page_content = data['Название показателя']
    del data['Название показателя']
    doc = Document(page_content=page_content, metadata=data)
    docs.append(doc)

store.add_documents(docs)

In [ ]:
from langchain.tools import tool

@tool
def search_tool(query : str):
    """
    Поиск по базе показателей
    """
    return store.similarity_search(query, k=10)

In [46]:
store.similarity_search('123')

[Document(id='0dc851f9-299e-4b70-b9bc-5b876a106767', metadata={'Раздел': 'Население', 'Код показателя': 8112021, 'Единица измерения': 'Человек'}, page_content='Число прибывших'),
 Document(id='3c977794-17ad-45ea-9c5f-13bfb45268bd', metadata={'Раздел': 'Организация отдыха, развлечений и культуры', 'Код показателя': 8017021, 'Единица измерения': 'Единица'}, page_content='Число цирков'),
 Document(id='fcda3259-9712-4464-ba0b-0c45f1971ed2', metadata={'Раздел': 'Розничная торговля и общественное питание', 'Код показателя': 8002007, 'Единица измерения': 'Единица'}, page_content='Число ярмарок'),
 Document(id='b20e5352-95b9-434d-86df-22eb498277b7', metadata={'Раздел': 'Население', 'Код показателя': 8112022, 'Единица измерения': 'Человек'}, page_content='Число выбывших')]

In [54]:
from src import Agent
from src.tools import ddgs_tool, lightrag_tool
from pydantic import BaseModel, Field
from typing import Literal

class Indicator(BaseModel):
    code : str = Field(description='Код показателя')
    name : str = Field(description='Название показателя')
    target : Literal['min', 'max'] = Field(description='Направление целевого значения (минимизация или максимизация)')

class Task(BaseModel):
    description : str = Field(description='Описание задачи')
    indicators : list[Indicator] = Field(description='Целевые показатели')

class Goal(BaseModel):
    description : str = Field(description='Описание цели')
    tasks : list[Task] = Field(description='Задачи пространственного развития')

class AgentResponse(BaseModel):
    external_factors : list[str] = Field(description='Внешние факторы развития')
    internal_factors : list[str] = Field(description='Внутренние факторы развития')
    mission : str = Field(description='Миссия МО')
    goals : list[Goal] = Field(description='Цели пространственного развития')

AGENT_PROMPT = """
На основе результатов аналитического этапа сформулируй:
- внешние и внутренние факторы развития;
- миссию МО;
- цели и задачи пространственного развития МО;
- систему целевых показателей.
---
ПРАВИЛА РАБОТЫ:
- Используй инструменты для поиска доступных показателей.
- Не придумывай показатели. Бери их из списка.
- Для каждого показателя укажи примерное целевое значение. 
"""

agent = Agent(system_prompt=AGENT_PROMPT, tools=[search_tool], response_format=AgentResponse, debug=True)
res = agent.invoke(str(data))

[values] {'messages': [HumanMessage(content='{\'диагностика экономического и пространственного состояния МО\': {\'социально-экономический анализ (демография, производительность труда, структура и динамика ВГП, рынок труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность)\': \'На основе анализа доступных источников представлю социально-экономический анализ Гатчинского муниципального округа Ленинградской области.\\n\\n## СОЦИАЛЬНО-ЭКОНОМИЧЕСКИЙ АНАЛИЗ ГАТЧИНСКОГО МУНИЦИПАЛЬНОГО ОКРУГА\\n\\n### ДЕМОГРАФИЯ\\n\\n**Численность населения:** По состоянию на 1 января 2024 года численность населения Гатчинского муниципального округа составляет 261 522 человека, что составляет 12,85% от общей численности населения Ленинградской области¹. \\n\\n**Структура населения:** Из общей численности сельское население составляет 89 832 человека¹. В округе расположено 234 сельских населённых пункта, крупнейшими из которых являются Новый Свет, Большие Колпаны, Малое Верево².\\n\\n**Городское

In [55]:
res.model_dump()

{'external_factors': ['Статус административного центра Ленинградской области - перенос высших органов власти в Гатчину создает предпосылки для ускоренного развития',
  'Планируемое строительство КАД-2 от трассы М-11 «Нева» до деревни Кузьмолово - улучшение транспортной доступности и инвестиционной привлекательности',
  'Включение Гатчины в проект «Умный город» - возможности для цифровизации и модернизации городской среды',
  'Близость к Санкт-Петербургу - потенциал для развития агломерационных связей и маятниковой миграции',
  'Реализация национальных проектов «Жилье и городская среда», «Безопасные качественные дороги» - дополнительные источники финансирования инфраструктуры',
  'Развитие индустриальных парков в Ленинградской области - возможности для размещения новых производств на территории округа'],
 'internal_factors': ['Высокий уровень автомобилизации (300 автомобилей на 1000 жителей) при недостаточной пропускной способности улично-дорожной сети',
  'Наличие историко-культурного 